In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

In [ ]:
session = get_active_session()

df_apartment = session.table("RENT_DB.RAW.RAW_RENT_APARTMENTS").to_pandas()
df_basements = session.table("RENT_DB.RAW.RAW_RENT_BASEMENTS").to_pandas()
df_condos = session.table("RENT_DB.RAW.RAW_RENT_CONDOS").to_pandas()
df_houses = session.table("RENT_DB.RAW.RAW_RENT_HOUSES").to_pandas()

In [ ]:
def profile_raw_table(session, table_name: str) -> pd.DataFrame:
    df = session.table(f"RENT_DB.RAW.{table_name}").to_pandas()
    profile = pd.DataFrame({
        "column":   df.columns,
        "dtype":    df.dtypes.values,
        "nulls":    df.isnull().sum().values,
        "distinct": df.nunique().values,
        "sample":   [df[c].dropna().iloc[0] if len(df[c].dropna()) else None for c in df.columns],
    })
    return profile

In [ ]:
raw_tables = [
    'RAW_RENT_APARTMENTS' , 
    'RAW_RENT_BASEMENTS' , 
    'RAW_RENT_CONDOS' , 
    'RAW_RENT_HOUSES']

In [ ]:
for tbl in raw_tables:
    print(f"\n── {tbl} ─────────────────────────────")
    display(profile_raw_table(session, tbl))

In [ ]:
tables = {
    "RAW_RENT_APARTMENTS": "apartment",
    "RAW_RENT_BASEMENTS": "basement",
    "RAW_RENT_CONDOS": "condo",
    "RAW_RENT_HOUSES": "house",
}

dfs = []
for table_name, property_type in tables.items():
    temp_df = session.table(f"RENT_DB.RAW.{table_name}").to_pandas()
    temp_df["PROPERTY_TYPE"] = property_type
    dfs.append(temp_df)

df_all = pd.concat(dfs, ignore_index=True)
df_all.shape

In [ ]:
df_all.sample(5)

In [ ]:
CREATE SCHEMA IF NOT EXISTS RENT_DB.ANALYTICS;

In [ ]:
snowpark_df = session.create_dataframe(df_all)
snowpark_df.write.mode("overwrite").save_as_table("RENT_DB.ANALYTICS.ALL_RENTALS")